In [1]:
%pip install -q transformers torch

import json
import os
import random
import glob
import logging
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TextClassificationPipeline,
)

CARPETA_DATA = 'data'
CARPETA_SALIDA = 'data_processed'

os.makedirs(CARPETA_SALIDA, exist_ok=True)

configuracion_medios = {
    '20minutos.json': {'Internacional': 20, 'Nacional': 20, 'Cultura': 20},
    'ABC.json': {'Internacional': 20, 'Nacional': 20, 'Cultura': 20},
    'elconfidencial.json': {'Internacional': 20, 'Nacional': 20, 'Cultura': 20},
    'elDiario.json': {'Internacional': 20, 'Nacional': 20, 'Cultura': 20},
    'elhuffpost.json': {'Internacional': 20, 'Nacional': 20, 'Cultura': 20},
    'lavanguardia.json': {'Internacional': 20, 'Nacional': 20, 'Cultura': 20},
    'okdiario.json': {'Internacional': 20, 'Nacional': 20, 'Cultura': 20},
    'RTVE.json': {'Internacional': 20, 'Nacional': 20, 'Cultura': 20},
    'mediterraneodigital.json': {'Internacional': 30, 'Nacional': 30}
}

noticias_finales = []

for nombre_archivo, distribucion in configuracion_medios.items():
    ruta_archivo = os.path.join(CARPETA_DATA, nombre_archivo)
    
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        noticias_medio = json.load(f)
            
    noticias_por_categoria = {cat: [] for cat in distribucion.keys()}
    
    for noticia in noticias_medio:
        cat = noticia.get("Categoría")
        if cat in distribucion:
            noticias_por_categoria[cat].append(noticia)
            
    for cat, cantidad in distribucion.items():
        disponibles = noticias_por_categoria[cat]
        
        cantidad_a_extraer = min(cantidad, len(disponibles)) 
        seleccionadas = random.sample(disponibles, cantidad_a_extraer)
        noticias_finales.extend(seleccionadas)

archivos_existentes = glob.glob(os.path.join(CARPETA_SALIDA, 'conjunto_noticias_*.json'))
numeros_existentes = []

for archivo in archivos_existentes:
    nombre_base = os.path.basename(archivo)
    try:
        num_str = nombre_base.replace('conjunto_noticias_', '').replace('.json', '')
        numeros_existentes.append(int(num_str))
    except ValueError:
        pass

siguiente_numero = max(numeros_existentes) + 1 if numeros_existentes else 1
nombre_salida = f'conjunto_noticias_{siguiente_numero}.json'
ruta_salida = os.path.join(CARPETA_SALIDA, nombre_salida)

_CLICKBAIT_LABEL = "Clickbait"
log = logging.getLogger(__name__)

def _cargar_pipeline(device: int) -> TextClassificationPipeline:
    tokenizer = AutoTokenizer.from_pretrained("taniwasl/clickbait_es")
    model = AutoModelForSequenceClassification.from_pretrained("taniwasl/clickbait_es")
    return TextClassificationPipeline(
        task="text-classification",
        model=model,
        tokenizer=tokenizer,
        max_length=25,
        truncation=True,
        add_special_tokens=True,
        device=device,
    )

def clasificar_clickbait(noticias_lista, guardar_en=None, batch_size=32, device=-1, verbose=True):
    if not noticias_lista:
        raise ValueError("La lista de noticias está vacía.")

    pipe = _cargar_pipeline(device)
    titulos = [n["Título"] for n in noticias_lista]
    resultados = []
    
    for i in range(0, len(titulos), batch_size):
        preds = pipe(titulos[i : i + batch_size])
        resultados.extend(preds)

    for noticia, pred in zip(noticias_lista, resultados):
        raw = pred["label"]
        score = round(pred["score"], 4)

        noticia["cb"] = (raw == _CLICKBAIT_LABEL)
        noticia["cb_score"] = score
        noticia["cb_label"] = raw

    if verbose:
        cb_total = sum(1 for n in noticias_lista if n["cb"])
        print(f"\n{'─'*50}")
        print(f" Total noticias procesadas : {len(noticias_lista)}")
        print(f" Clickbait                 : {cb_total} ({100*cb_total/len(noticias_lista):.1f}%)")
        print(f" No clickbait              : {len(noticias_lista)-cb_total} ({100*(len(noticias_lista)-cb_total)/len(noticias_lista):.1f}%)")
        print(f"{'─'*50}\n")

    if guardar_en is not None:
        out = Path(guardar_en)
        out.parent.mkdir(parents=True, exist_ok=True)
        with out.open("w", encoding="utf-8") as f:
            json.dump(noticias_lista, f, ensure_ascii=False, indent=4)
        print(f"Resultado guardado en: {out}")

    return noticias_lista

_ = clasificar_clickbait(noticias_finales, guardar_en=ruta_salida)

Note: you may need to restart the kernel to use updated packages.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


──────────────────────────────────────────────────
 Total noticias procesadas : 540
 Clickbait                 : 131 (24.3%)
 No clickbait              : 409 (75.7%)
──────────────────────────────────────────────────

Resultado guardado en: data_processed\conjunto_noticias_2.json
